# 키 확인

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
# print(os.getenv('KEY'))

# 실행확인

In [64]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
load_dotenv()
key=os.getenv('KEY')

url = "https://data-dbg.krx.co.kr/svc/apis/idx/krx_dd_trd"
params = {
    "AUTH_KEY": key,
    "basDd": 20251229,   # 기준일자
    'IDX_NM' : 'KRX 300',
    "REQ_TYPE": "json"
}

response = requests.get(url, params=params) 
data = response.json()

# JSON 파싱
if "OutBlock_1" in data and data["OutBlock_1"]:
    df = pd.json_normalize(data["OutBlock_1"])
    df = df[["BAS_DD", "IDX_NM", "CLSPRC_IDX", "ACC_TRDVOL"]]
    print(df)
else:
    print("주식시장 휴무:")

      BAS_DD             IDX_NM CLSPRC_IDX ACC_TRDVOL
0   20251128         코리아 밸류업 지수    1616.85   44522437
1   20251128            KRX TMI    2410.28  799218768
2   20251128            KRX 300    2522.66  173724922
3   20251128        KRX 중대형 TMI    2422.23  320849932
4   20251128         KRX 중형 TMI    1621.14  147125010
5   20251128         KRX 소형 TMI    1503.61  352057525
6   20251128        KRX 초소형 TMI    5286.67  126311311
7   20251128            KTOP 30   10895.70   36413120
8   20251128            KRX 100    8663.27   69910298
9   20251128            KRX 자동차    2331.62   17722548
10  20251128            KRX 반도체    5805.84   72964688
11  20251128           KRX 헬스케어    4989.18   31020906
12  20251128             KRX 은행    1285.76    6925545
13  20251128          KRX 에너지화학    2655.04    5342008
14  20251128             KRX 철강    2473.88    2894984
15  20251128           KRX 방송통신     763.38    1222515
16  20251128             KRX 건설     789.41   15026877
17  20251128             KRX

# 실행코드

In [76]:
import requests
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os
load_dotenv()
key=os.getenv('KEY')

url = "https://data-dbg.krx.co.kr/svc/apis/idx/krx_dd_trd"

start_date = datetime(2025, 9, 30)   # 시작일
end_date   = datetime.today()  # 종료일

all_data = []
not_in_data = []

while start_date <= end_date:
    date_str = start_date.strftime("%Y%m%d")  # 'YYYYMMDD' 형식
    params = {
        "AUTH_KEY": key,
        "basDd": date_str,
        "REQ_TYPE": "json"
    }
    response = requests.get(url, params=params)
    data = response.json()

    if "OutBlock_1" in data and data["OutBlock_1"]:
        # 원하는 IDX_NM만 필터링
        for row in data["OutBlock_1"]:
            if row["IDX_NM"] in ["KRX 건설", "KRX 자동차", "KRX 헬스케어"]:
                all_data.append(row)
    
    # 휴무일을 확인할 필요가 있는 경우
    #else:
        ## 휴무일, 장 중에는 데이터가 없음
        #print(f"{date_str} 데이터 없음")
        ## 날짜 저장 리스트
        #not_in_data.append(date_str)

    start_date += timedelta(days=1)

# DataFrame으로 변환 후 필요한 열만 추출
df = pd.DataFrame(all_data)[["BAS_DD", "IDX_NM", "CLSPRC_IDX", "ACC_TRDVOL"]]
df.columns = ["날짜", "종목명", "종가(백만원)", "거래량(천주)"]
df.sort_values(by=['날짜','종목명'],ascending=[True,True], inplace=True)
df.reset_index(drop=True, inplace=True)

check_data = df.iloc[:,-2].copy()
KRX_items = set(df.iloc[:,1])

data_count = len(check_data)
KRX_count = len(KRX_items)
date_count = int(data_count/KRX_count)

up_down_list = []
# 전체데이터에 날짜당 아이템 갯수를 나누어 묶음들을 구함
for date_idx in range(date_count):
    # 묶음 내부의 값들의 증감치 추가
    # 첫 묶음은 비교대상이 없기에 '시작일'로 지정
    if date_idx == 0:
        for start_idx in range(KRX_count):
            up_down_list.append('시작일')
    else:
        # 현재 묶음과 이전 묶음의 각각의 아이템의 크기를 비교 
        for udw_idx in range(KRX_count):
            if check_data[date_idx*3 + udw_idx] > check_data[(date_idx-1)*3 + udw_idx]:
                up_down_list.append('증가')
            elif check_data[date_idx*3 + udw_idx] < check_data[(date_idx-1)*3 + udw_idx]:
                up_down_list.append('감소')
            else:
                up_down_list.append('변동없음')

df['전일대비'] = up_down_list
df.to_csv('data/주가지수크롤링.csv')
print('작업완료')

작업완료


In [ ]:
# 확인코드

In [80]:
df.sample(5)

,날짜,종목명,종가(백만원),거래량(천주),전일대비
75,20251111,KRX 건설,786.81,7566701,감소
36,20251023,KRX 건설,819.59,9592099,증가
96,20251120,KRX 건설,772.78,5822130,증가
85,20251114,KRX 자동차,2280.69,8029623,감소
10,20251010,KRX 자동차,1947.34,15487431,감소
